# ch06 形态学与轮廓（练习版）

> 按提示补全 `____`，跑通所有 assert。`coins` / `cg` / `cbw`（全局 Otsu 二值图）/
> `noisy`（400 个白噪点）已在 setup 里备好；`K3` / `K5` / `ascii_bin` / `biggest` /
> `bbox_of` / `diff` 脚手架可用。
>
> 注意：挖空块内每条语句保持**单行**；控制流写在挖空块之外。
> 题目真值都来自实跑（opencv 5.0.0），照提示写就能对上。

In [ ]:
from pathlib import Path

import warnings

import numpy as np

warnings.simplefilter("ignore")

import cv2

DATA = Path("data")
CL = DATA / "classic"
HE = DATA / "helmet"

In [ ]:
coins = cv2.imread(str(CL / "coins.png"))
cg = cv2.cvtColor(coins, cv2.COLOR_BGR2GRAY)
cg_blur = cv2.GaussianBlur(cg, (5, 5), 0)
otsu_t, cbw = cv2.threshold(cg_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# 400 个白噪点（固定种子，结果可复现）
rng = np.random.default_rng(0)
noisy = cbw.copy()
ys = rng.integers(0, noisy.shape[0], 400)
xs = rng.integers(0, noisy.shape[1], 400)
noisy[ys, xs] = 255

print("coins", coins.shape, "| gray", cg.shape, "| Otsu T", otsu_t, "| cbw 前景", int((cbw > 0).sum()))
print("cg 灰度范围", int(cg.min()), "~", int(cg.max()), "-> 全局单一阈值在这张图上有光照不均风险")
print("noisy 前景", int((noisy > 0).sum()), "| 外轮廓",
      len(cv2.findContours(noisy, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]))

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
K3 = np.ones((3, 3), np.uint8)
K5 = np.ones((5, 5), np.uint8)


def ascii_bin(m, mark="##"):
    """把小尺寸二值图渲染成文本（0 -> ".."，非 0 -> mark）。"""
    return "\n".join("".join(mark if v else ".." for v in row) for row in m)


def biggest(m):
    """返回面积最大的外轮廓（没有轮廓则返回 None）。"""
    cs = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
    return max(cs, key=cv2.contourArea) if cs else None


def bbox_of(m):
    """返回最大外轮廓的 boundingRect（空图返回 None）。"""
    c = biggest(m)
    return None if c is None else cv2.boundingRect(c)


def diff(a, b):
    """两张二值图的逐像素最大差（0 表示完全相同）。"""
    return int(np.abs(a.astype(np.int32) - b.astype(np.int32)).max())


def close_fill(m, k):
    """用 k x k 全 1 核做闭运算后的前景像素数（用来观察「洞有没有被填上」）。"""
    return int((cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((k, k), np.uint8)) > 0).sum())


print("脚手架就绪：K3 / K5 / ascii_bin / biggest / bbox_of / diff / close_fill")

## 任务 1：腐蚀 / 膨胀的几何

方块、细线、外框各变化多少？

In [ ]:
sq = np.zeros((9, 9), np.uint8)
sq[3:6, 3:6] = 255
line = np.zeros((9, 9), np.uint8)
line[:, 4] = 255
# TODO(1)：腐蚀 = 邻域取 min（白区被啃瘦），膨胀 = 邻域取 max（白区变胖）
# 提示：都用 K3；各量 (原图, 腐蚀后, 膨胀后) 三个前景像素数，注意 1px 细线会被啃光
sq_px = ____
line_px = ____
bb_sq, bb_er, bb_di = ____
print("3x3 方块 (原, 腐蚀, 膨胀) 前景:", sq_px)
print("1px 竖线 (原, 腐蚀, 膨胀) 前景:", line_px)
print("方块外框: 原", bb_sq, "| 腐蚀后", bb_er, "| 膨胀后", bb_di)
print("腐蚀后剩下的像素坐标:", np.argwhere(cv2.erode(sq, K3) > 0).tolist())
print("解读: 3x3 方块腐蚀后就剩中心 1 个点（9 -> 1），膨胀成 5x5（25）—— 核 3x3 把边界各吃掉 1 圈。"
      "1px 细线的腐蚀结果是 **0**（整条消失），膨胀成 3x9=27。"
      "所以「腐蚀掉噪点」和「腐蚀掉细线」是同一个机制，选核大小就是在选你能忍受多细的损失")

assert sq_px == (9, 1, 25) and line_px == (9, 0, 27)
assert (bb_sq, bb_er, bb_di) == ((3, 3, 3, 3), (4, 4, 1, 1), (2, 2, 5, 5))

## 任务 2：结构元的形状

`RECT` / `ELLIPSE` / `CROSS` 的有效面积。

In [ ]:
k_rect3 = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
k_ell3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
k_cross3 = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
k_ell5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
k_rect5 = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
# TODO(2)：结构元形状也是旋钮：RECT / ELLIPSE / CROSS 的「有效面积」并不一样
# 提示：数每个核的 1 元素个数；再判断 ELLIPSE3 与 CROSS3 是否逐元素相同；最后看方块用 ELLIPSE3 膨胀的面积
k_sizes = ____
k_same = ____
di_ell = ____
print("核的 1 元素个数 (RECT3, ELLIPSE3, ELLIPSE5, RECT5):", k_sizes)
print("ELLIPSE3 与 CROSS3 逐元素相同?", k_same)
print("ELLIPSE5 行内元素数:", k_ell5.sum(axis=1).tolist())
print("方块用 ELLIPSE3 膨胀后面积:", di_ell, "（RECT3 是 25）")
print("解读: 3x3 时 ELLIPSE 退化成 CROSS（只有 5 个 1），比 RECT 的 9 个少 —— "
      "所以「膨胀 3x3」的默认含义取决于核形状，用 ELLIPSE 得到的是十字形邻域，"
      "方块只能长成 21 像素的「圆角方块」而不是 25。Ellipse 核在圆目标上更自然，"
      "但形状一变，同样的 ksize 结果就不可比")

assert k_sizes == (9, 5, 17, 25) and k_same is True and di_ell == 21

## 任务 3：迭代 n 次 ≡ 一次大核

矩形核下的等价性，要逐像素为 0 的证据。

In [ ]:
# TODO(3)：迭代 n 次 3x3 是否等价于「一次 (2n+1)x(2n+1) 矩形核」？（要逐像素为 0 的证据）
# 提示：比 erode(cbw, K3, iterations=2) 与 erode(cbw, K5)；再比 iterations=3 与 7x7 全 1 核；用 diff() 对账
er_32 = ____
er_51 = ____
di_32 = ____
di_51 = ____
er_33 = ____
d_er2, d_di2 = ____
d_er3 = ____
px_iter = ____
print("erode 3x3x2 vs 5x5x1: 前景", int((er_32 > 0).sum()), "vs", int((er_51 > 0).sum()), "| 逐像素差", d_er2)
print("dilate 3x3x2 vs 5x5x1: 前景", int((di_32 > 0).sum()), "vs", int((di_51 > 0).sum()), "| 逐像素差", d_di2)
print("erode 3x3x3 vs 7x7x1 逐像素差:", d_er3)
print("(er_32, di_32, er_33) 前景:", px_iter)
print("解读: 矩形核下 **迭代 2 次 3x3 == 一次 5x5**，3 次 == 一次 7x7（差全为 0）—— "
      "因为 5x5 方块 = 3x3 方块 ⊕ 3x3 方块，而「核的膨胀」对应「运算的叠加」。"
      "实战价值：想开很大的核，用 iterations 比造大核便宜（大核是 O(k²) 元素，"
      "迭代是 O(n) 次 3x3）。但换成 ELLIPSE 核这个等式就不成立了")

assert d_er2 == 0 and d_di2 == 0 and d_er3 == 0
assert px_iter == (37867, 58379, 33150)

## 任务 4：开运算与闭运算

先与手写组合对账，再看两者之间的差异。

In [ ]:
# TODO(4)：开运算 = 先腐蚀后膨胀（去小白点）；闭运算 = 先膨胀后腐蚀（填小黑洞）
# 提示：cv2.morphologyEx(cbw, cv2.MORPH_OPEN / MORPH_CLOSE, K3)，再与手写的 erode→dilate / dilate→erode 对账
op3 = ____
cl3 = ____
d_op = ____
d_cl = ____
px_morph = ____
d_opcl = ____
print("(cbw, open3, close3) 前景:", px_morph)
print("open3 == dilate(erode) 逐像素差:", d_op, "| close3 == erode(dilate) 逐像素差:", d_cl)
print("open3 与 close3 的最大差:", d_opcl)
print("解读: 开运算与手写 erode→dilate 逐像素差 0，闭运算与 dilate→erode 差 0 —— 定义完全对得上。"
      "但 open3 与 close3 彼此差 255：**它们是两个方向相反的算子**，"
      "一个往「小」的方向修（侵蚀掉小的白结构），一个往「大」的方向修（填掉小的黑结构），不能互换")

assert px_morph == (48069, 47961, 48300)
assert d_op == 0 and d_cl == 0 and d_opcl == 255

## 任务 5：实战 400 个白噪点

开运算 vs 闭运算，谁的轮廓数更低、谁的前景更多？

In [ ]:
# TODO(5)：实战：开运算能不能把 400 个白噪点的轮廓数压回去？闭运算呢？
# 提示：对 noisy 做 OPEN / CLOSE，分别数 RETR_EXTERNAL 轮廓数与前景像素数（都用 K3）
n_noisy = ____
noisy_op = ____
noisy_cl = ____
noisy_res = ____
print("噪声图轮廓数 原:", n_noisy, "| open 后 (轮廓, 前景):", noisy_res[:2], "| close 后:", noisy_res[2:])
print("解读: 开运算用来**去掉比核小的白点**（400 个噪点把轮廓从 34 顶到 252，"
      "open3 后回到 28，比原始 cbw 的 34 还干净 —— 顺带把硬币内部的亮斑也修了）；"
      "闭运算用来**填掉比核小的黑洞**，但它同样会把孤立的噪点**连成一片**"
      "（close 后轮廓 224，前景反涨到 48625）。"
      "记住口诀：**开去白噪点、闭填黑洞**，用反了就是把噪声放大")

assert n_noisy == 252 and noisy_res == (28, 47965, 224, 48625)

## 任务 6：开闭的尺度边界

洞要多小才填得上？线要多粗才不被抹掉？

In [ ]:
plate = np.zeros((25, 25), np.uint8)
plate[5:20, 5:20] = 255
hole3 = plate.copy()
hole3[11:14, 11:14] = 0
hole5 = plate.copy()
hole5[10:15, 10:15] = 0
thin1 = np.zeros((60, 60), np.uint8)
thin1[30, 5:55] = 255
thin1[10:50, 30] = 255
thin5 = np.zeros((60, 60), np.uint8)
thin5[28:33, 5:55] = 255
thin5[10:50, 28:33] = 255
# TODO(6)：闭运算能填多大的洞？开运算会干掉多细的线？（边界条件比结论重要）
# 提示：洞的对照：3x3 洞 / 5x5 洞 分别用 close3 / close5 / close7，看前景能否回到 225（=plate 面积）
# 提示：线的对照：1px 十字与 5px 十字各做一次 open3，看前景还剩多少
fill3 = ____
fill5 = ____
thin_px = ____
print("plate 前景:", int((plate > 0).sum()))
print("3x3 洞 + close3/5/7 后前景:", fill3)
print("5x5 洞 + close3/5/7 后前景:", fill5)
print("(1px 十字原, open3, 5px 十字原, open3):", thin_px)
print("解读: 结论很干脆 —— **洞的尺寸必须严格小于核尺寸才填得上**："
      "3x3 洞要 close5、5x5 洞要 close7，同尺寸的 close3/close5 都填不满。"
      "线也一样：1px 十字被 open3 **整条抹掉**（89 -> 0），5px 十字毫发无伤（425 -> 425）。"
      "所以形态学的所有「去噪」都是**按尺度筛结构**，没有一个既能保细节又只去噪声的核")

assert fill3 == (216, 225, 225) and fill5 == (200, 200, 225)
assert thin_px == (89, 0, 425, 425)

## 任务 7：梯度 / 顶帽 / 黑帽

三个差式定义 + 等式验证。

In [ ]:
# TODO(7)：三个派生算子：形态学梯度 / 顶帽 / 黑帽，分别等于哪两个基础运算的差？
# 提示：MORPH_GRADIENT / MORPH_TOPHAT / MORPH_BLACKHAT，再用 diff() 验证等式
mg = ____
th = ____
bh = ____
px_ops = ____
d_mg = ____
d_th = ____
d_bh = ____
th_max = ____
print("(gradient, tophat, blackhat) 前景:", px_ops)
print("gradient == dilate - erode 差:", d_mg)
print("tophat == src - open 差:", d_th, "| blackhat == close - src 差:", d_bh)
print("tophat 最大值:", th_max)
print("解读: 三个都是「差出来的」——梯度 = 膨胀 - 腐蚀（前景的边界带，等于对二值图做了一遍轮廓描边）；"
      "顶帽 = 原图 - 开运算（留下被开运算抹掉的**亮的细结构**：噪点、细线、高光）；"
      "黑帽 = 闭运算 - 原图（留下被填掉的**暗的细结构**：裂纹、划痕、孔洞）。"
      "**顶帽最实用的地方是「光照校正」**：开运算 ≈ 低频背景，原图减掉它就只剩高频细节")

assert px_ops == (10423, 108, 231) and th_max == 255
assert d_mg == 0 and d_th == 0 and d_bh == 0

## 任务 8：`findContours` 三件套

返回值个数 / 检索模式 / 点近似方法。

In [ ]:
# TODO(8)：findContours 的三件套：返回值个数 / 检索模式 / 点近似方法
# 提示：老版本返回 (image, contours, hierarchy) 三个值、新版本只有两个，先用 len() 确认你手上是几个
# 提示：再数 RETR_EXTERNAL / RETR_LIST / RETR_TREE 的轮廓数，最后数三种 CHAIN_APPROX 的点总数
fc_ret = ____
n_ret = ____
cnts_ext, hier_ext = ____
n_ext, hier_shape = ____
n_list = ____
n_tree = ____
ext_none = ____
ext_simp = ____
ext_tc89 = ____
pts = ____
print("cv2 版本:", cv2.__version__, "| findContours 返回元组长度:", n_ret)
print("RETR_EXTERNAL 轮廓数:", n_ext, "| hierarchy 形状:", hier_shape)
print("RETR_LIST:", n_list, "| RETR_TREE:", n_tree)
print("点总数 (APPROX_NONE, SIMPLE, TC89_L1):", pts)
print("解读: 1) OpenCV 3 返回 (image, contours, hierarchy) 3 个值，4/5 只返回 2 个 —— "
      "老教程里 `img, cnts, hier = cv2.findContours(...)` 在新版会直接 ValueError，"
      "跨版本代码要写 `cnts, hier = cv2.findContours(...)[-2:]`；"
      "2) hierarchy 恒为 (1, N, 4) 的 [next, prev, first_child, parent]；"
      "3) **APPROX_SIMPLE 省掉了共线的中间点**（3703 -> 1729 点，省 53%），"
      "算面积/周长时三种近似给出的答案是同一量级，但点数差 4 倍直接影响后续处理速度")

assert n_ret == 2 and n_ext == 30 and hier_shape == (1, 30, 4)
assert n_list == 33 and n_tree == 33 and pts == (3703, 1729, 833)

## 任务 9：轮廓的几何量

取一枚独立硬币，量面积、周长、圆度、凸性、`approxPolyDP`。

In [ ]:
# TODO(9)：轮廓的几何量：面积 / 周长 / 外接矩形 / 外接圆 / 凸包 / 多边形逼近
# 提示：按面积排序取**第 2 大**（第 1 大是顶行与过曝背景粘连的块），它就是一枚独立硬币
# 提示：周长用 cv2.arcLength(c, True)；圆度用 4πA/P²；凸性用 A / hull 面积
cs_size = ____
coin = ____
c_bbox = ____
c_area = ____
c_fill = ____
c_per, c_per_open = ____
c_circ = ____
(cx, cy), c_r = ____
hull = ____
c_solidity = ____
c_pt = ____
print("这枚硬币的 bbox:", c_bbox, "| 矩形面积:", c_bbox[2] * c_bbox[3])
print("contourArea:", c_area, "| 填充率 area/(w*h):", c_fill)
print("arcLength closed:", c_per, "| open:", c_per_open, "| 圆度 4πA/P²:", c_circ)
print("minEnclosingCircle 圆心:", (round(cx, 2), round(cy, 2)), "半径:", round(c_r, 4))
print("凸包点数:", len(hull), "（原轮廓", len(coin), "点）| 凸性 solidity:", c_solidity)
print("approxPolyDP 点数 (eps=0.001/0.002/0.005/0.01/0.02/0.05 * 周长):", c_pt)
print("解读: 硬币是圆，所以填充率只有 0.7585（圆内切于边长 65x61 的矩形，π/4≈0.785），"
      "圆度 4πA/P² 0.8871 已经接近 1；solidity 0.9836 说明轮廓几乎全是凸的（缺口很小）。"
      "arcLength(closed=True) 比 False 多 6.0 —— 它把首尾闭合的那段也算进去了，算周长必须 True。"
      "**approxPolyDP 的 eps 是「能容忍的最大偏离」**：0.005*P（约 1 px）就把 90 个点压成 18 个，"
      "0.05*P 只剩 5 个 —— 这就是「用几个点描述一个形状」的精度-体积旋钮")

assert c_bbox == (315, 156, 65, 61) and c_area == 3007.5 and c_fill == 0.7585
assert c_per == 206.4092 and c_per_open == 200.4092 and c_circ == 0.8871
assert (round(cx, 2), round(cy, 2), round(c_r, 4)) == (347.21, 186.8, 32.4343)
assert len(hull) == 34 and len(coin) == 90 and c_solidity == 0.9836
assert c_pt == (90, 62, 18, 13, 8, 5)

## 任务 10：连通域 vs 轮廓

两种口径能不能对上？差在哪？

In [ ]:
# TODO(10)：connectedComponentsWithStats：连通域计数与轮廓计数对不对得上？
# 提示：返回 (n, labels, stats, centroids)，n 把背景也算一个；面积列用 stats[:, cv2.CC_STAT_AREA]
n_cc, cc_lbl, cc_stats, cc_cent = ____
cc_area = ____
cc_kept = ____
cc_top6 = ____
cc_first_bbox = ____
print("connectedComponentsWithStats 返回 n =", n_cc, "（含背景），所以前景连通域", n_cc - 1, "个")
print("面积 >1000 的连通域:", cc_kept,
      "| 而轮廓（面积>1000）也是", sum(1 for c in cnts_ext if cv2.contourArea(c) > 1000), "个")
print("面积前 6:", cc_top6)
print("第 1 个前景连通域的 bbox:", cc_first_bbox, "| 质心:", [round(v, 2) for v in cc_cent[1].tolist()])
print("解读: 两者**在这种「实心块、无嵌套」的图上是等价的**，但语义不同："
      "连通域给的是**像素级**的标签图（labels 可以逐像素索引）+ 现成的面积/外接框/质心；"
      "轮廓给的是**边界点序列**（能画、能算周长、能逼近多边形）。"
      "所以「只要个数和面积」用 connectedComponentsWithStats 更快更省内存，"
      "「要形状」才用 findContours。注意 n 含背景，别把 n 当成目标个数")

assert n_cc == 31 and cc_kept == 22
assert cc_top6 == [13319, 3097, 2665, 2423, 2136, 1949]
assert cc_first_bbox == [0, 0, 331, 79]

## 任务 11：综合——硬币到底几枚

三种口径各数一遍，再和目检对账。

In [ ]:
# TODO(11)：综合：硬币到底几枚？—— 三种口径各数一遍，再和人工目检对账
# 提示：口径 A：按面积阈值扫描；口径 B：先用顶帽（51x51 椭圆核）校正光照再数；口径 C：霍夫圆交叉验证
cs_all = ____
n_by_thr = ____
k_top = ____
top51 = ____
top_bw = ____
top_mask = ____
top_cs = ____
n_tophat = ____
cir = ____
n_hough = ____
n_manual = ____
top_max, top_mean = ____
print("口径 A（全局 Otsu + close3，面积阈值 >10 / >1000 / >1500）:", n_by_thr)
print("面积最大的块 bbox:", cv2.boundingRect(cs_all[0]), "面积:", round(cv2.contourArea(cs_all[0]), 1))
print("口径 B（顶帽 51x51 校正光照 + 面积>800）:", n_tophat, "| 顶帽图 max", top_max, "mean", top_mean)
print("口径 C（霍夫圆 param2=25）:", n_hough)
print("人工目检:", n_manual)
print("解读: 同一张图给出 22 / 23 / 24 三个数 —— 这就是「计数」的真相："
      "**数字是「预处理 + 形态学 + 面积阈值」三者的联合产物**。"
      "22 的根源是最大块 bbox(0,0,331,79)、面积 13113.5 ≈ 4 枚硬币：顶行硬币与过曝背景"
      "（图上方背景整体亮于 Otsu 阈值 104）粘成一片；顶帽把低频背景先减掉，就修正到 23；"
      "霍夫圆带「圆」的几何先验，给出正确的 24。"
      "**所以任何计数结论都要写清口径、并且必须目检**——只交一个数字、不交口径，"
      "在评审/答辩时是站不住的")

assert n_by_thr == (22, 22, 10)
assert (n_tophat, n_hough, n_manual) == (23, 24, 24)
assert top_max == 226 and top_mean == 47.2838

## 自查清单

- [ ] 能说出腐蚀 / 膨胀各是邻域的 min 还是 max
- [ ] 知道为什么 1px 细线会被 `erode(K3)` 抹掉
- [ ] 记得 `ELLIPSE(3,3)` 就是 `CROSS(3,3)`，膨胀方块只长 21 px
- [ ] 能解释「迭代 2 次 3×3 == 5×5」成立的前提
- [ ] 能用一句话说清开运算和闭运算各干什么、用反了什么后果
- [ ] 知道闭运算填洞的尺寸条件（严格小于核）
- [ ] 能默写 `gradient / tophat / blackhat` 的差式定义
- [ ] 知道顶帽怎么用来做光照校正
- [ ] 记得 `findContours` 在 OpenCV 4/5 返回 2 个值
- [ ] 说得清 `RETR_EXTERNAL` 与 `RETR_TREE` 差在哪些轮廓
- [ ] 知道算周长必须 `arcLength(c, True)`
- [ ] 能背出圆度与凸性的公式
- [ ] 知道「个数/面积」用连通域、「形状」用轮廓
- [ ] 能解释为什么同一张图能数出 22 / 23 / 24